In [1]:
import os
import pandas as pd
import torch
import matplotlib.pyplot as plt
import numpy as np
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

#if os.path.isdir(path):
#    print(f"Directory contents for {path}:")
#    for filename in os.listdir(path):
#        print(f"- {filename}")
#else:
#    print(f"Downloaded file: {path}")

In [2]:
os.environ["CUDA_VISIBLE_DEVICES"] = "1" 

testOrTrain = 'train'
rootPath = "/home/stu14/s16/acm7552/.cache/kagglehub/datasets/ravidussilva/real-ai-art/versions/5/Real_AI_SD_LD_Dataset/train"
#rootPath = "/home/stu14/s16/acm7552/"

In [3]:
#rootPath = "your/path/here" # Replace with your actual path
contents = os.listdir(rootPath)
contents

['AI_SD_renaissance',
 'AI_LD_art_nouveau',
 'realism',
 'impressionism',
 'AI_SD_surrealism',
 'AI_SD_impressionism',
 'AI_LD_post_impressionism',
 'AI_SD_realism',
 'AI_SD_post_impressionism',
 'AI_SD_expressionism',
 'art_nouveau',
 'AI_SD_romanticism',
 'AI_LD_realism',
 'expressionism',
 'surrealism',
 'AI_LD_renaissance',
 'ukiyo_e',
 'AI_SD_art_nouveau',
 'AI_LD_ukiyo-e',
 'AI_LD_expressionism',
 'AI_LD_baroque',
 'romanticism',
 'AI_LD_surrealism',
 'AI_LD_impressionism',
 'post_impressionism',
 'baroque',
 'renaissance',
 'AI_SD_ukiyo-e',
 'AI_LD_romanticism',
 'AI_SD_baroque']

In [5]:
transform = transforms.Compose([
    transforms.Resize((256, 256)), # real images should be this resolution, AI images will need resizing
    transforms.ToTensor()
])# convert to tensor])

In [6]:
#ImageFolder is a livesaver here
image_dataset = datasets.ImageFolder(root=rootPath, transform=transform)

# need binary label mapping. by default InageFolder will make every folder into a class. I dont want that
# this checks the name of the folder. if it starts with "AI_", it goes in the AI class.
ai_class_indices = [
    idx for cls, idx in image_dataset.class_to_idx.items()
    if "AI_" in cls
]

# replace the labels now
new_samples = []
for path, original_label in image_dataset.samples:
    if original_label in ai_class_indices:
        new_label = 1  # AI
    else:
        new_label = 0  # Real
    new_samples.append((path, new_label))

# Replace everything cleanly
image_dataset.samples = new_samples
image_dataset.targets = [label for _, label in new_samples]
image_dataset.classes = ["Real", "AI"]
image_dataset.class_to_idx = {"Real": 0, "AI": 1}

dataloader = DataLoader(image_dataset, batch_size=32, shuffle=True, num_workers=4)

print("Classes:", image_dataset.classes)
print("Class to index mapping:", image_dataset.class_to_idx)
print(f"Number of images found: {len(image_dataset)}")

for images, labels in dataloader:
    print("Batch images shape:", images.shape)
    print("Batch labels:", labels)
    break 

Classes: ['Real', 'AI']
Class to index mapping: {'Real': 0, 'AI': 1}
Number of images found: 155015
Batch images shape: torch.Size([32, 3, 256, 256])
Batch labels: tensor([1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1,
        0, 1, 1, 1, 1, 1, 1, 1])


Cool. This should be good enough for my purposes, though I should also make some helper functions that can display or save images as desired.